
<div  style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://raw.githubusercontent.com/derar-alhussein/Databricks-Certified-Data-Engineer-Associate/main/Includes/images/bookstore_schema.png" alt="Databricks Learning" style="width: 600">
</div>

# 0 Extracción de datos desde archivos con Spark SQL

En este notebook aprenderemos a extraer datos directamente desde archivos utilizando **Spark SQL en Databricks**.

🗂️ Dataset: Bookstore

Para esta demostración trabajaremos con un dataset de una librería, compuesto por tres tablas principales:

- `customers`
- `books`
- `orders`

A continuación, utilizaremos un notebook auxiliar que descargará y copiará estos datos en el sistema de archivos de Databricks.

📄 Formato de los datos

Los datos de la tabla `customers` están almacenados en formato **JSON**, lo que nos permitirá ver cómo Spark puede leer directamente este tipo de archivos.

In [0]:
%run "./Copy-Datasets"

Data Catalog: workspacedemofull02


Copying books-cdc/ ...
Copying books-csv-new/ ...
Copying books-csv/ ...
Copying books-streaming/ ...
Copying customers-json-new/ ...
Copying customers-json/ ...
Copying orders-json-raw/ ...
Copying orders-json-streaming/ ...
Copying orders-new/ ...
Copying orders-raw/ ...
Copying orders-streaming/ ...
Copying orders/ ...


## 0.1 Explorando los datos

Vamos a listar los archivos dentro del directorio `customers`.

Podemos observar que hay varios archivos en formato **JSON** (en este caso, seis archivos) dentro de este directorio.

🔍 Lectura de los datos

A continuación, vamos a leer estos archivos para explorar su contenido utilizando Spark.

In [0]:
%python
files = dbutils.fs.ls(f"/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/")
display(files)

path,name,size,modificationTime
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_001.json,export_001.json,79378,1776100398000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_002.json,export_002.json,80001,1776100398000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_003.json,export_003.json,79781,1776100399000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_004.json,export_004.json,79976,1776100400000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_005.json,export_005.json,79727,1776100400000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_006.json,export_006.json,53243,1776100401000


# 1 Lectura de un archivo JSON

Para consultar un único archivo JSON en Spark SQL, indicamos la ruta completa del archivo directamente en la consulta.

Es importante tener en cuenta que la ruta debe escribirse ``utilizando **backticks**`` y no comillas simples.

---

📊 Explorando la estructura de los datos

Una vez leídos los datos, podemos observar las distintas columnas disponibles en la tabla:

- identificador del cliente
- correo electrónico
- información de perfil, almacenada en formato JSON
- marca temporal de última actualización

---

👀 Vista previa de los datos

La vista previa nos permite comprobar tanto la estructura como el contenido del archivo.

En este ejemplo, se muestran los **300 registros** disponibles en el archivo fuente, lo que facilita una validación rápida antes de continuar con el procesamiento.

In [0]:
SELECT * 
FROM json.`/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_001.json`

customer_id,email,profile,updated
C00001,dabby2y@japanpost.jp,"{""first_name"":""Dniren"",""last_name"":""Abby"",""gender"":""Female"",""address"":{""street"":""768 Mesta Terrace"",""city"":""Annecy"",""country"":""France""}}",2021-12-14T23:15:43.375Z
C00002,eabbysc1@github.com,"{""first_name"":""Etti"",""last_name"":""Abbys"",""gender"":""Female"",""address"":{""street"":""1748 Vidon Plaza"",""city"":""Varge Mondar"",""country"":""Portugal""}}",2021-12-14T23:15:43.375Z
C00003,rabelovd1@wikispaces.com,"{""first_name"":""Ronnie"",""last_name"":""Abelov"",""gender"":""Male"",""address"":{""street"":""363 Randy Park"",""city"":""San Celestio"",""country"":""Philippines""}}",2021-12-14T23:15:43.375Z
C00004,rabels9g@behance.net,"{""first_name"":""Ray"",""last_name"":""Abels"",""gender"":""Female"",""address"":{""street"":""613 Lyons Way"",""city"":""Oudtshoorn"",""country"":""South Africa""}}",2021-12-14T23:15:43.375Z
C00005,sabendrothin@cargocollective.com,"{""first_name"":""Shanon"",""last_name"":""Abendroth"",""gender"":""Female"",""address"":{""street"":""30292 Manufacturers Junction"",""city"":""Ani-e"",""country"":""Philippines""}}",2021-12-14T23:15:43.375Z
C00006,null,"{""first_name"":""Norman"",""last_name"":""Abernethy"",""gender"":""Male"",""address"":{""street"":""9292 Oxford Center"",""city"":""Gibara"",""country"":""Cuba""}}",2021-12-14T23:15:43.375Z
C00007,sabrahmson3h@blinklist.com,"{""first_name"":""Skell"",""last_name"":""Abrahmson"",""gender"":""Male"",""address"":{""street"":""90941 Hallows Park"",""city"":""Huarong Chengguanzhen"",""country"":""China""}}",2021-12-14T23:15:43.375Z
C00008,dacheson2h@mapy.cz,"{""first_name"":""Darsey"",""last_name"":""Acheson"",""gender"":""Non-binary"",""address"":{""street"":""29579 Grim Plaza"",""city"":""Dārayyā"",""country"":""Syria""}}",2021-12-14T23:15:43.375Z
C00009,fackwoodji@gravatar.com,"{""first_name"":""Fredrick"",""last_name"":""Ackwood"",""gender"":""Male"",""address"":{""street"":""67 Dunning Plaza"",""city"":""Santo Domingo"",""country"":""Cuba""}}",2021-12-14T23:15:43.375Z
C00010,null,"{""first_name"":""Doralynne"",""last_name"":""Adamkiewicz"",""gender"":""Female"",""address"":{""street"":""84126 Glendale Center"",""city"":""Ugep"",""country"":""Nigeria""}}",2021-12-14T23:15:43.375Z


## 1.1 Lectura de múltiples archivos

También es posible consultar varios archivos al mismo tiempo utilizando **caracteres comodín (wildcards)**.

Por ejemplo, podemos leer todos los archivos JSON cuyo nombre comienza por un mismo prefijo, como `export_`, sin necesidad de especificar cada archivo individualmente.

---

⚙️ Comportamiento por defecto

Cuando consultamos múltiples archivos, la vista previa muestra por defecto un número limitado de registros.

En concreto, se visualizarán los **primeros 1.000 registros**, lo cual permite explorar rápidamente los datos sin cargar el dataset completo.

In [0]:
SELECT * FROM json.`/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_*.json`

customer_id,email,profile,updated
C00301,thomas.lane@gmail.com,"{""first_name"":""Thomas"",""last_name"":""Lane"",""gender"":""Male"",""address"":{""street"":""06 Boulevard Victor Hugo"",""city"":""Paris"",""country"":""France""}}",2021-12-14T23:15:43.375Z
C00302,ocolegatele@blogger.com,"{""first_name"":""Odilia"",""last_name"":""Colegate"",""gender"":""Female"",""address"":{""street"":""07 Sommers Parkway"",""city"":""Lyon"",""country"":""France""}}",2021-12-14T23:15:43.375Z
C00303,acolledged2@nbcnews.com,"{""first_name"":""Andros"",""last_name"":""Colledge"",""gender"":""Male"",""address"":{""street"":""342 Katie Center"",""city"":""Gort"",""country"":""Ireland""}}",2021-12-14T23:15:43.375Z
C00304,null,"{""first_name"":""Iver"",""last_name"":""Collet"",""gender"":""Male"",""address"":{""street"":""12126 Union Point"",""city"":""Iguape"",""country"":""Brazil""}}",2021-12-14T23:15:43.375Z
C00305,pcollier5r@cmu.edu,"{""first_name"":""Page"",""last_name"":""Collier"",""gender"":""Male"",""address"":{""street"":""3 Farragut Lane"",""city"":""Berlin"",""country"":""Germany""}}",2021-12-14T23:15:43.375Z
C00306,null,"{""first_name"":""Tally"",""last_name"":""Collins"",""gender"":""Male"",""address"":{""street"":""4 Hovde Park"",""city"":""Cairo"",""country"":""Egypt""}}",2021-12-14T23:15:43.375Z
C00307,lcollocottcm@t-online.de,"{""first_name"":""Leupold"",""last_name"":""Collocott"",""gender"":""Male"",""address"":{""street"":""917 Stephen Circle"",""city"":""Dzerzhinskiy"",""country"":""Russia""}}",2021-12-14T23:15:43.375Z
C00308,icolloughfa@prweb.com,"{""first_name"":""Inesita"",""last_name"":""Collough"",""gender"":""Female"",""address"":{""street"":""7910 Delladonna Street"",""city"":""Osoyoos"",""country"":""Canada""}}",2021-12-14T23:15:43.375Z
C00309,jcollymore4n@pcworld.com,"{""first_name"":""Joelle"",""last_name"":""Collymore"",""gender"":""Female"",""address"":{""street"":""19 Dayton Court"",""city"":""Yidu"",""country"":""China""}}",2021-12-14T23:15:43.375Z
C00310,gcolnetef@japanpost.jp,"{""first_name"":""Goldi"",""last_name"":""Colnet"",""gender"":""Female"",""address"":{""street"":""710 Knutson Place"",""city"":""Suso"",""country"":""Philippines""}}",2021-12-14T23:15:43.375Z


## 1.2 Lectura de un directorio completo

Además, también podemos consultar un **directorio completo de archivos**, siempre que todos los archivos compartan el mismo formato y esquema.

En este caso, en lugar de indicar un archivo específico, simplemente especificamos la ruta del directorio.

---

📊 Resultado

De esta forma, conseguimos leer de manera conjunta todos los datos contenidos en ese directorio, tratándolos como un único dataset.

In [0]:
SELECT * FROM json.`/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json`

customer_id,email,profile,updated
C00301,thomas.lane@gmail.com,"{""first_name"":""Thomas"",""last_name"":""Lane"",""gender"":""Male"",""address"":{""street"":""06 Boulevard Victor Hugo"",""city"":""Paris"",""country"":""France""}}",2021-12-14T23:15:43.375Z
C00302,ocolegatele@blogger.com,"{""first_name"":""Odilia"",""last_name"":""Colegate"",""gender"":""Female"",""address"":{""street"":""07 Sommers Parkway"",""city"":""Lyon"",""country"":""France""}}",2021-12-14T23:15:43.375Z
C00303,acolledged2@nbcnews.com,"{""first_name"":""Andros"",""last_name"":""Colledge"",""gender"":""Male"",""address"":{""street"":""342 Katie Center"",""city"":""Gort"",""country"":""Ireland""}}",2021-12-14T23:15:43.375Z
C00304,null,"{""first_name"":""Iver"",""last_name"":""Collet"",""gender"":""Male"",""address"":{""street"":""12126 Union Point"",""city"":""Iguape"",""country"":""Brazil""}}",2021-12-14T23:15:43.375Z
C00305,pcollier5r@cmu.edu,"{""first_name"":""Page"",""last_name"":""Collier"",""gender"":""Male"",""address"":{""street"":""3 Farragut Lane"",""city"":""Berlin"",""country"":""Germany""}}",2021-12-14T23:15:43.375Z
C00306,null,"{""first_name"":""Tally"",""last_name"":""Collins"",""gender"":""Male"",""address"":{""street"":""4 Hovde Park"",""city"":""Cairo"",""country"":""Egypt""}}",2021-12-14T23:15:43.375Z
C00307,lcollocottcm@t-online.de,"{""first_name"":""Leupold"",""last_name"":""Collocott"",""gender"":""Male"",""address"":{""street"":""917 Stephen Circle"",""city"":""Dzerzhinskiy"",""country"":""Russia""}}",2021-12-14T23:15:43.375Z
C00308,icolloughfa@prweb.com,"{""first_name"":""Inesita"",""last_name"":""Collough"",""gender"":""Female"",""address"":{""street"":""7910 Delladonna Street"",""city"":""Osoyoos"",""country"":""Canada""}}",2021-12-14T23:15:43.375Z
C00309,jcollymore4n@pcworld.com,"{""first_name"":""Joelle"",""last_name"":""Collymore"",""gender"":""Female"",""address"":{""street"":""19 Dayton Court"",""city"":""Yidu"",""country"":""China""}}",2021-12-14T23:15:43.375Z
C00310,gcolnetef@japanpost.jp,"{""first_name"":""Goldi"",""last_name"":""Colnet"",""gender"":""Female"",""address"":{""street"":""710 Knutson Place"",""city"":""Suso"",""country"":""Philippines""}}",2021-12-14T23:15:43.375Z


## 1.3 Número de clientes

A continuación, podemos calcular cuántos clientes hay en el dataset.

El resultado nos muestra que contamos con un total de **1.700 clientes**.

In [0]:
SELECT count(*) FROM json.`/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json`

count(*)
1700


## 1.4 Identificación del archivo de origen

Cuando trabajamos con múltiples archivos, puede resultar muy útil identificar de qué archivo proviene cada registro.

Para ello, Spark ofrece una función integrada que permite añadir una columna con el **nombre del archivo de origen**.

---

🛠️ Utilidad

Esta información es especialmente útil en escenarios como:

- depuración de datos
- identificación de errores en archivos concretos
- trazabilidad del origen de la información

Especial incapie en ``file_modification_time`` — timestamp de última modificación, muy útil en pipelines incrementales para saber cuándo llegó el fichero

---

📊 Resultado

Además de las columnas originales del dataset, obtenemos una nueva columna que indica el archivo fuente de cada registro, lo que facilita enormemente el análisis y la resolución de posibles problemas.

In [0]:
SELECT *,
    _metadata.file_path       AS source_file,
    _metadata.file_name       AS file_name,
    _metadata.file_size       AS file_size_bytes,
    _metadata.file_modification_time AS last_modified,
    _metadata.file_block_length AS block_length
FROM json.`/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json`;

customer_id,email,profile,updated,source_file,file_name,file_size_bytes,last_modified,block_length
C00301,thomas.lane@gmail.com,"{""first_name"":""Thomas"",""last_name"":""Lane"",""gender"":""Male"",""address"":{""street"":""06 Boulevard Victor Hugo"",""city"":""Paris"",""country"":""France""}}",2021-12-14T23:15:43.375Z,dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_002.json,export_002.json,80001,2026-04-13T17:13:18.000Z,80001
C00302,ocolegatele@blogger.com,"{""first_name"":""Odilia"",""last_name"":""Colegate"",""gender"":""Female"",""address"":{""street"":""07 Sommers Parkway"",""city"":""Lyon"",""country"":""France""}}",2021-12-14T23:15:43.375Z,dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_002.json,export_002.json,80001,2026-04-13T17:13:18.000Z,80001
C00303,acolledged2@nbcnews.com,"{""first_name"":""Andros"",""last_name"":""Colledge"",""gender"":""Male"",""address"":{""street"":""342 Katie Center"",""city"":""Gort"",""country"":""Ireland""}}",2021-12-14T23:15:43.375Z,dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_002.json,export_002.json,80001,2026-04-13T17:13:18.000Z,80001
C00304,null,"{""first_name"":""Iver"",""last_name"":""Collet"",""gender"":""Male"",""address"":{""street"":""12126 Union Point"",""city"":""Iguape"",""country"":""Brazil""}}",2021-12-14T23:15:43.375Z,dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_002.json,export_002.json,80001,2026-04-13T17:13:18.000Z,80001
C00305,pcollier5r@cmu.edu,"{""first_name"":""Page"",""last_name"":""Collier"",""gender"":""Male"",""address"":{""street"":""3 Farragut Lane"",""city"":""Berlin"",""country"":""Germany""}}",2021-12-14T23:15:43.375Z,dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_002.json,export_002.json,80001,2026-04-13T17:13:18.000Z,80001
C00306,null,"{""first_name"":""Tally"",""last_name"":""Collins"",""gender"":""Male"",""address"":{""street"":""4 Hovde Park"",""city"":""Cairo"",""country"":""Egypt""}}",2021-12-14T23:15:43.375Z,dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_002.json,export_002.json,80001,2026-04-13T17:13:18.000Z,80001
C00307,lcollocottcm@t-online.de,"{""first_name"":""Leupold"",""last_name"":""Collocott"",""gender"":""Male"",""address"":{""street"":""917 Stephen Circle"",""city"":""Dzerzhinskiy"",""country"":""Russia""}}",2021-12-14T23:15:43.375Z,dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_002.json,export_002.json,80001,2026-04-13T17:13:18.000Z,80001
C00308,icolloughfa@prweb.com,"{""first_name"":""Inesita"",""last_name"":""Collough"",""gender"":""Female"",""address"":{""street"":""7910 Delladonna Street"",""city"":""Osoyoos"",""country"":""Canada""}}",2021-12-14T23:15:43.375Z,dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_002.json,export_002.json,80001,2026-04-13T17:13:18.000Z,80001
C00309,jcollymore4n@pcworld.com,"{""first_name"":""Joelle"",""last_name"":""Collymore"",""gender"":""Female"",""address"":{""street"":""19 Dayton Court"",""city"":""Yidu"",""country"":""China""}}",2021-12-14T23:15:43.375Z,dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_002.json,export_002.json,80001,2026-04-13T17:13:18.000Z,80001
C00310,gcolnetef@japanpost.jp,"{""first_name"":""Goldi"",""last_name"":""Colnet"",""gender"":""Female"",""address"":{""street"":""710 Knutson Place"",""city"":""Suso"",""country"":""Philippines""}}",2021-12-14T23:15:43.375Z,dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_002.json,export_002.json,80001,2026-04-13T17:13:18.000Z,80001


#2 Lectura de archivos como texto

Otra opción interesante es utilizar formatos de tipo **texto**, que permiten consultar cualquier archivo basado en texto, como:

- JSON  
- CSV  
- TSV  
- TXT  

---

📊 Representación de los datos

Al leer los archivos de esta forma, cada línea del fichero se interpreta como una fila independiente, con una única columna de tipo string llamada **value**.

---

🛠️ Casos de uso

Este enfoque resulta especialmente útil cuando:

- los datos están corruptos o mal formateados  
- el esquema no es consistente  
- necesitamos aplicar lógica personalizada para parsear el contenido  

En estos casos, podemos trabajar directamente con el texto y aplicar funciones propias para extraer la información necesaria.

In [0]:
SELECT * FROM text.`/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json`

value
"{""customer_id"":""C00301"",""email"":""thomas.lane@gmail.com"",""profile"":""{\""first_name\"":\""Thomas\"",\""last_name\"":\""Lane\"",\""gender\"":\""Male\"",\""address\"":{\""street\"":\""06 Boulevard Victor Hugo\"",\""city\"":\""Paris\"",\""country\"":\""France\""}}"",""updated"":""2021-12-14T23:15:43.375Z""}"
"{""customer_id"":""C00302"",""email"":""ocolegatele@blogger.com"",""profile"":""{\""first_name\"":\""Odilia\"",\""last_name\"":\""Colegate\"",\""gender\"":\""Female\"",\""address\"":{\""street\"":\""07 Sommers Parkway\"",\""city\"":\""Lyon\"",\""country\"":\""France\""}}"",""updated"":""2021-12-14T23:15:43.375Z""}"
"{""customer_id"":""C00303"",""email"":""acolledged2@nbcnews.com"",""profile"":""{\""first_name\"":\""Andros\"",\""last_name\"":\""Colledge\"",\""gender\"":\""Male\"",\""address\"":{\""street\"":\""342 Katie Center\"",\""city\"":\""Gort\"",\""country\"":\""Ireland\""}}"",""updated"":""2021-12-14T23:15:43.375Z""}"
"{""customer_id"":""C00304"",""profile"":""{\""first_name\"":\""Iver\"",\""last_name\"":\""Collet\"",\""gender\"":\""Male\"",\""address\"":{\""street\"":\""12126 Union Point\"",\""city\"":\""Iguape\"",\""country\"":\""Brazil\""}}"",""updated"":""2021-12-14T23:15:43.375Z""}"
"{""customer_id"":""C00305"",""email"":""pcollier5r@cmu.edu"",""profile"":""{\""first_name\"":\""Page\"",\""last_name\"":\""Collier\"",\""gender\"":\""Male\"",\""address\"":{\""street\"":\""3 Farragut Lane\"",\""city\"":\""Berlin\"",\""country\"":\""Germany\""}}"",""updated"":""2021-12-14T23:15:43.375Z""}"
"{""customer_id"":""C00306"",""profile"":""{\""first_name\"":\""Tally\"",\""last_name\"":\""Collins\"",\""gender\"":\""Male\"",\""address\"":{\""street\"":\""4 Hovde Park\"",\""city\"":\""Cairo\"",\""country\"":\""Egypt\""}}"",""updated"":""2021-12-14T23:15:43.375Z""}"
"{""customer_id"":""C00307"",""email"":""lcollocottcm@t-online.de"",""profile"":""{\""first_name\"":\""Leupold\"",\""last_name\"":\""Collocott\"",\""gender\"":\""Male\"",\""address\"":{\""street\"":\""917 Stephen Circle\"",\""city\"":\""Dzerzhinskiy\"",\""country\"":\""Russia\""}}"",""updated"":""2021-12-14T23:15:43.375Z""}"
"{""customer_id"":""C00308"",""email"":""icolloughfa@prweb.com"",""profile"":""{\""first_name\"":\""Inesita\"",\""last_name\"":\""Collough\"",\""gender\"":\""Female\"",\""address\"":{\""street\"":\""7910 Delladonna Street\"",\""city\"":\""Osoyoos\"",\""country\"":\""Canada\""}}"",""updated"":""2021-12-14T23:15:43.375Z""}"
"{""customer_id"":""C00309"",""email"":""jcollymore4n@pcworld.com"",""profile"":""{\""first_name\"":\""Joelle\"",\""last_name\"":\""Collymore\"",\""gender\"":\""Female\"",\""address\"":{\""street\"":\""19 Dayton Court\"",\""city\"":\""Yidu\"",\""country\"":\""China\""}}"",""updated"":""2021-12-14T23:15:43.375Z""}"
"{""customer_id"":""C00310"",""email"":""gcolnetef@japanpost.jp"",""profile"":""{\""first_name\"":\""Goldi\"",\""last_name\"":\""Colnet\"",\""gender\"":\""Female\"",\""address\"":{\""street\"":\""710 Knutson Place\"",\""city\"":\""Suso\"",\""country\"":\""Philippines\""}}"",""updated"":""2021-12-14T23:15:43.375Z""}"


In [0]:
-- Extraer campos específicos sin leer el schema completo
SELECT 
    get_json_object(value, '$.customer_id') AS customer_id,
    get_json_object(value, '$.email')       AS email,
    get_json_object(value, '$.country')     AS country
FROM text.`/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json`;

customer_id,email,country
C00301,thomas.lane@gmail.com,null
C00302,ocolegatele@blogger.com,null
C00303,acolledged2@nbcnews.com,null
C00304,null,null
C00305,pcollier5r@cmu.edu,null
C00306,null,null
C00307,lcollocottcm@t-online.de,null
C00308,icolloughfa@prweb.com,null
C00309,jcollymore4n@pcworld.com,null
C00310,gcolnetef@japanpost.jp,null


#3 Lectura de archivos binarios

Además, también podemos trabajar con archivos en formato **binario**, lo que nos permite extraer directamente los bytes de cada archivo.

---

📊 Información disponible

Al leer archivos como binarios, obtenemos información como:

- ruta del archivo  
- fecha de última modificación  
- tamaño del archivo  
- contenido en formato binario  

---

🛠️ Utilidad

Este enfoque es útil cuando necesitamos:

- procesar archivos no estructurados  
- trabajar con imágenes, documentos u otros formatos binarios  
- aplicar lógica personalizada sobre el contenido del archivo  

El contenido se representa como una secuencia de bytes, lo que permite un tratamiento más flexible de los datos.

In [0]:
SELECT * FROM binaryFile.`/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json`

path,modificationTime,length,content
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_002.json,2026-03-18T06:46:01.000Z,80001,...
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_004.json,2026-03-18T06:46:02.000Z,79976,...
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_003.json,2026-03-18T06:46:02.000Z,79781,...
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_005.json,2026-03-18T06:46:03.000Z,79727,...
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_001.json,2026-03-18T06:46:01.000Z,79378,...
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json/export_006.json,2026-03-18T06:46:03.000Z,53243,...



# 4 Lectura de datos en formato CSV

A continuación, pasamos a trabajar con los datos de libros, que están almacenados en formato **CSV**.

El proceso es similar al visto anteriormente: utilizamos una consulta para leer directamente los archivos, indicando en este caso el formato CSV.

---

🔄 Cambio de formato

A diferencia del ejemplo anterior con JSON, aquí simplemente cambiamos el formato de lectura para adaptarlo a archivos CSV, manteniendo la misma lógica de trabajo.

In [0]:
SELECT * FROM csv.`/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv`

_c0
book_id;title;author;category;price
B07;The Hundred-Page Machine Learning;Andriy Burkov;Computer Science;33
B08;Quantum Computing for Everyone;Chris Bernhardt;Computer Science;41
B09;Advanced Data Structures;Peter Brass;Computer Science;24
book_id;title;author;category;price
B10;Beginning Database Design Solutions;Rod Stephens;Computer Science;44
B11;Business Intelligence for Dummies;Swain Scheps;Computer Science;38
B12;Big Data in Practice;Bernard Marr;Computer Science;30
book_id;title;author;category;price
B01;The Soul of a New Machine;Tracy Kidder;Computer Science;49


## 4.1 Limitaciones al leer archivos CSV directamente

Hemos conseguido leer los datos, pero el resultado no es correcto.

Podemos observar que:

- la fila de cabecera se está interpretando como una fila más de datos  
- todos los valores aparecen cargados en una única columna  
- esto ocurre porque el delimitador real del archivo no es una coma, sino un **punto y coma**  

---

📌 Por qué ocurre esto

La lectura directa de archivos funciona especialmente bien con formatos **auto descriptivos**, es decir, formatos que ya incluyen información suficiente sobre su estructura, como por ejemplo:

- JSON  
- Parquet  

Sin embargo, en formatos como **CSV**, donde el esquema no viene definido de forma explícita, esta aproximación no suele ser suficiente.

En estos casos, necesitamos una forma de proporcionar configuración adicional, como:

- los nombres de las columnas  
- los tipos de datos  
- si existe cabecera  
- el delimitador utilizado en el archivo  

---

🏗️ Crear una tabla sobre archivos externos

Una solución habitual es crear una tabla utilizando la cláusula **USING**, lo que nos permite definir una tabla sobre una fuente de datos externa, como un conjunto de archivos CSV.

Para ello, debemos indicar:

- el esquema de la tabla, incluyendo nombres y tipos de columnas  
- el formato de los archivos  
- si los archivos incluyen cabecera  
- el delimitador utilizado para separar los campos  
- la ubicación donde se encuentran los archivos  

---

✅ Resultado

Una vez proporcionada esta información, la tabla queda correctamente creada y los datos pueden interpretarse de forma adecuada.

In [0]:
%python
import re

# Get current user
user = spark.sql("SELECT current_user()").first()[0]

# Clean username
user_clean = re.sub(r"@.*", "", user)
user_clean = user_clean.replace(".", "_").replace("-", "_")

table_name = "books_csv"
location = f"s3://mi-bucket-publico-javier-2026/tables/{user_clean}/{table_name}"

print(user_clean)
print(location)

test_data_jm
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/books_csv


In [0]:
%python
source_path = "/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv"
target_path = location

df = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .csv(source_path)
)

(df.write
   .mode("overwrite")
   .option("header", "true")
   .option("delimiter", ";")
   .csv(target_path))


spark.sql(f"""
CREATE TABLE books_csv
(
  book_id STRING,
  title STRING,
  author STRING,
  category STRING,
  price DOUBLE
)
USING CSV
OPTIONS (
  header = "true",
  delimiter = ";"
)
LOCATION '{target_path}'
""")

DataFrame[]

In [0]:
SELECT * FROM books_csv

book_id,title,author,category,price
B07,The Hundred-Page Machine Learning,Andriy Burkov,Computer Science,33.0
B08,Quantum Computing for Everyone,Chris Bernhardt,Computer Science,41.0
B09,Advanced Data Structures,Peter Brass,Computer Science,24.0
B10,Beginning Database Design Solutions,Rod Stephens,Computer Science,44.0
B11,Business Intelligence for Dummies,Swain Scheps,Computer Science,38.0
B12,Big Data in Practice,Bernard Marr,Computer Science,30.0
B01,The Soul of a New Machine,Tracy Kidder,Computer Science,49.0
B02,Learning JavaScript Design Patterns,Addy Osmani,Computer Science,28.0
B03,Make Your Own Neural Network,Tariq Rashid,Computer Science,35.0
B04,Robot Dynamics and Control,Mark W. Spong,Computer Science,20.0


## 4.2 Consideraciones  al trabajar con archivos CSV

Es importante tener en cuenta una limitación clave al trabajar con **archivos CSV como fuente de datos**.

A diferencia de formatos como Parquet o JSON, los CSV **no contienen información de esquema**, por lo que Spark interpreta los datos **en función del orden de las columnas**.

Por ello, es fundamental asegurarse de que:

- **El orden de las columnas no cambie** si se añaden nuevos archivos al directorio origen.
- Todos los archivos mantengan la **misma estructura y orden de campos**.

Spark siempre cargará los datos y aplicará los **nombres de columnas y tipos de datos en el orden definido durante la creación de la tabla**.

Si el orden cambia, los datos podrían **mapearse incorrectamente entre columnas**, generando errores difíciles de detectar.

## 4.3 Explorando los metadatos de la tabla externa

Ejecutemos el comando **`DESCRIBE EXTENDED`** para obtener información relevante sobre nuestra tabla.

Dado que el provider es **`CSV`** y no **`DELTA`**, **no se ha generado ningún fichero nuevo** en el storage. Cada vez que se consulte esta tabla, Spark leerá los CSV originales en tiempo real aplicando la configuración registrada en el metastore (`header`, `delimiter`, etc.).

Esto contrasta con una tabla Delta, donde los datos se escriben físicamente como ficheros Parquet junto con un transaction log (`_delta_log`).

In [0]:
DESCRIBE EXTENDED books_csv

col_name,data_type,comment
book_id,string,null
title,string,null
author,string,null
category,string,null
price,double,null
,,
# Detailed Table Information,,
Catalog,workspace,
Database,default,
Table,books_csv,


# 5 Limitaciones al trabajar con fuentes externas (CSV)

Veamos ahora el impacto de trabajar con una tabla que **no es una data table gestionada por Delta Lake**, sino una tabla que apunta a **archivos CSV externos**.

Cuando trabajamos con **data tables (por ejemplo, Delta Lake)**, contamos con múltiples garantías y funcionalidades, como:
- Consistencia en las lecturas
- Control de versiones
- Acceso siempre a la **versión más reciente de los datos**

Sin embargo, al trabajar con **fuentes externas como CSV**, perdemos estas garantías.

Por ejemplo:
- Las tablas Delta garantizan que siempre consultamos la **última versión de los datos**
- Mientras que una tabla basada en CSV puede **no reflejar inmediatamente cambios recientes**, ya que depende directamente de los archivos físicos y su estado en el almacenamiento

Para entender mejor este comportamiento, vamos a realizar un experimento:
1. Añadiremos un nuevo archivo CSV al directorio origen
2. Observaremos cómo impacta esto en nuestra tabla

Primero, comprobemos **cuántos archivos CSV existen actualmente en el directorio**.

In [0]:
%python
dataset_bookstore="/Volumes/workspacedemofull02/default/bookstore_dataset"
files = dbutils.fs.ls(f"{dataset_bookstore}/books-csv")
display(files)

path,name,size,modificationTime
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv/export_001.csv,export_001.csv,238,1776100382000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv/export_002.csv,export_002.csv,237,1776100383000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv/export_003.csv,export_003.csv,240,1776100383000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv/export_004.csv,export_004.csv,223,1776100384000


Estado actual del directorio:

Actualmente, podemos observar que en el directorio existen **4 archivos CSV**.

Este será nuestro punto de referencia antes de añadir nuevos datos y analizar cómo se comporta la tabla al trabajar con una fuente externa.

## 5.1 Añadiendo nuevos datos al directorio CSV

Para esta demostración utilizaremos la **API de DataFrame de Spark**, que nos permite **escribir datos en distintos formatos**, como por ejemplo CSV.

La idea es sencilla:

1. Leeremos la tabla **`books_csv`** que acabamos de crear  
2. Reescribiremos esos mismos datos en **un nuevo archivo CSV adicional**  
3. Guardaremos ese archivo en el **mismo directorio origen**

De esta forma, estaremos simulando un escenario real en el que **se añaden nuevos ficheros al dataset**, lo que nos permitirá analizar cómo responde la tabla al trabajar con una fuente externa basada en archivos CSV.

In [0]:
%python

(spark.read
        .table("books_csv")
      .write
        .mode("append")
        .format("csv")
        .option('header', 'true')
        .option('delimiter', ';')
        .save(f"{dataset_bookstore}/books-csv"))

## 5.2 Verificación tras añadir nuevos archivos

Volvamos a comprobar el contenido del directorio para ver cuántos archivos CSV existen ahora.

Como podemos observar, **se han añadido nuevos archivos CSV al directorio**, que han sido generados por Spark al escribir los datos.

Esto confirma que el dataset ya no está compuesto únicamente por los archivos originales, sino que ahora incluye **archivos adicionales**, simulando un escenario real de crecimiento de datos en el Data Lake.

In [0]:
%python
files = dbutils.fs.ls(f"{dataset_bookstore}/books-csv")
display(files)

path,name,size,modificationTime
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv/_SUCCESS,_SUCCESS,0,1776102253000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv/_committed_2920725937665517037,_committed_2920725937665517037,380,1776102253000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv/_started_2920725937665517037,_started_2920725937665517037,0,1776102252000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv/export_001.csv,export_001.csv,238,1776100382000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv/export_002.csv,export_002.csv,237,1776100383000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv/export_003.csv,export_003.csv,240,1776100383000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv/export_004.csv,export_004.csv,223,1776100384000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv/part-00000-tid-2920725937665517037-5d9b03b7-8c27-4d69-a81a-e5b676fcdc45-241-1-c000.csv,part-00000-tid-2920725937665517037-5d9b03b7-8c27-4d69-a81a-e5b676fcdc45-241-1-c000.csv,246,1776102252000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv/part-00001-tid-2920725937665517037-5d9b03b7-8c27-4d69-a81a-e5b676fcdc45-242-1-c000.csv,part-00001-tid-2920725937665517037-5d9b03b7-8c27-4d69-a81a-e5b676fcdc45-242-1-c000.csv,244,1776102252000
dbfs:/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv/part-00002-tid-2920725937665517037-5d9b03b7-8c27-4d69-a81a-e5b676fcdc45-243-1-c000.csv,part-00002-tid-2920725937665517037-5d9b03b7-8c27-4d69-a81a-e5b676fcdc45-243-1-c000.csv,243,1776102252000


In [0]:
SELECT COUNT(*) FROM books_csv

COUNT(*)
12


In [0]:
REFRESH TABLE books_csv

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6309019916867124>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'REFRESH TABLE books_csv\n')

File /databricks/python/lib/python3.11/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:152, in SqlMagic.sql(self, line, cell)
    148     raise Exception(
    149         "Cannot run %sql command because spark conne

In [0]:
SELECT COUNT(*) FROM books_csv

COUNT(*)
0


# 6 Creación de tablas Delta a partir de fuentes externas

Como hemos visto, trabajar con tablas basadas en archivos como CSV tiene ciertas limitaciones.

Para superar estas limitaciones, podemos crear **tablas Delta**, cargando datos desde fuentes externas mediante sentencias como **`CREATE TABLE AS SELECT` (CTAS)**.

En este caso, creamos y poblamos la tabla **`customers`** utilizando datos provenientes de una consulta sobre archivos JSON.

Este proceso:
1. **Lee los datos desde los archivos JSON**
2. **Los carga en una tabla gestionada por Delta Lake**
3. Genera automáticamente la estructura de archivos necesaria (Parquet + `_delta_log`)

Al revisar los metadatos de la tabla, podemos comprobar que:
- Se trata de una tabla de tipo **Delta**
- Además, es una **tabla gestionada (managed table)**, ya que no hemos especificado una ubicación externa

In [0]:
CREATE TABLE customers AS
SELECT * FROM json.`/Volumes/workspacedemofull02/default/bookstore_dataset/customers-json`;

DESCRIBE EXTENDED customers;

col_name,data_type,comment
customer_id,string,null
email,string,null
profile,string,null
updated,string,null
,,
# Delta Statistics Columns,,
Column Names,"customer_id, email, profile, updated",
Column Selection Method,first-32,
,,
# Detailed Table Information,,


Inferencia automática de esquema en CTAS

Además, podemos observar que el **schema ha sido inferido automáticamente** a partir de los resultados de la consulta.

Esto ocurre porque las sentencias **`CREATE TABLE AS SELECT` (CTAS)**:
- **Infieren automáticamente el esquema** basándose en los datos devueltos por la query
- **No permiten definir el esquema manualmente**

Por este motivo, las sentencias CTAS son especialmente útiles para:
- **Ingestar datos desde fuentes externas**
- Trabajar con formatos que ya tienen un **esquema bien definido**, como:
  - **Parquet**
  - Otras tablas existentes

En estos casos, el proceso de creación de la tabla es más sencillo, ya que **no es necesario declarar explícitamente las columnas y tipos de datos**.

## 6.1 Limitaciones de CTAS con ciertos formatos

Además, las sentencias **`CREATE TABLE AS SELECT` (CTAS)** tienen otra limitación importante:  
**no permiten especificar opciones adicionales de lectura de archivos**.

Esto supone un problema especialmente al trabajar con **archivos CSV**, donde es habitual necesitar opciones como:
- `header = true`
- `delimiter`
- `inferSchema`

Debido a esta limitación, aunque hayamos creado correctamente una **tabla Delta**, es posible que **los datos no se hayan interpretado correctamente** (por ejemplo, todo en una sola columna o con tipos incorrectos).

Por tanto, CTAS es muy útil para formatos con esquema definido (como Parquet), pero **no es la mejor opción para ingestión de CSV**, donde suele ser necesario un mayor control sobre las opciones de lectura.

In [0]:
CREATE TABLE books_unparsed AS
SELECT * FROM csv.`/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv`;

SELECT * FROM books_unparsed;

_c0
book_id;title;author;category;price
B10;Beginning Database Design Solutions;Rod Stephens;Computer Science;44
B11;Business Intelligence for Dummies;Swain Scheps;Computer Science;38
B12;Big Data in Practice;Bernard Marr;Computer Science;30
book_id;title;author;category;price
B01;The Soul of a New Machine;Tracy Kidder;Computer Science;49
B02;Learning JavaScript Design Patterns;Addy Osmani;Computer Science;28
B03;Make Your Own Neural Network;Tariq Rashid;Computer Science;35
book_id;title;author;category;price
B07;The Hundred-Page Machine Learning;Andriy Burkov;Computer Science;33


## 6.2 Solución: uso de vista temporal para definir opciones de lectura

Para corregir este problema, primero necesitamos **referenciar los archivos de forma que podamos especificar opciones de lectura** (como `header`, `delimiter`, etc.).

Para ello, creamos una **vista temporal**, donde sí podemos definir estas opciones correctamente al leer los archivos.

A continuación, utilizamos esta **vista temporal como origen en una sentencia `CREATE TABLE AS SELECT` (CTAS)**.

De esta forma:
1. Aplicamos correctamente las **opciones de lectura sobre los archivos**
2. Utilizamos esa vista ya estructurada como input
3. Creamos finalmente una **tabla Delta con los datos correctamente interpretados**

Este patrón es especialmente útil al trabajar con **formatos como CSV**, donde es necesario controlar cómo se leen los datos antes de cargarlos en una tabla Delta.

In [0]:
CREATE TEMP VIEW books_tmp_vw
   (book_id STRING, title STRING, author STRING, category STRING, price DOUBLE)
USING CSV
OPTIONS (
  path = "/Volumes/workspacedemofull02/default/bookstore_dataset/books-csv/export_*.csv",
  header = "true",
  delimiter = ";"
);

CREATE TABLE books AS
  SELECT * FROM books_tmp_vw;
  
SELECT * FROM books

book_id,title,author,category,price
B10,Beginning Database Design Solutions,Rod Stephens,Computer Science,44.0
B11,Business Intelligence for Dummies,Swain Scheps,Computer Science,38.0
B12,Big Data in Practice,Bernard Marr,Computer Science,30.0
B01,The Soul of a New Machine,Tracy Kidder,Computer Science,49.0
B02,Learning JavaScript Design Patterns,Addy Osmani,Computer Science,28.0
B03,Make Your Own Neural Network,Tariq Rashid,Computer Science,35.0
B07,The Hundred-Page Machine Learning,Andriy Burkov,Computer Science,33.0
B08,Quantum Computing for Everyone,Chris Bernhardt,Computer Science,41.0
B09,Advanced Data Structures,Peter Brass,Computer Science,24.0
B04,Robot Dynamics and Control,Mark W. Spong,Computer Science,20.0


Resultado de la carga de datos

La tabla se ha creado correctamente y los datos han sido cargados como se esperaba.

Observa que en este caso estamos recuperando **12 registros**.  
Esto se debe a que hemos utilizado un **carácter comodín (`*`) en la ruta de los archivos**.

El uso del comodín permite que Spark lea **múltiples archivos que coinciden con el patrón especificado**, en lugar de un único fichero, lo que facilita la ingestión de datos cuando estos están distribuidos en varios archivos dentro de un directorio.

## 6.3 Verificación final de la tabla Delta

Por último, revisemos la **metadata de nuestra tabla Delta**.

Al hacerlo, podemos confirmar que:
- Se trata efectivamente de una **tabla Delta**
- Los datos han sido **extraídos desde los archivos CSV**
- Y han sido **almacenados correctamente en la ubicación de la tabla**

Con esto completamos el proceso de ingestión desde archivos externos hacia una **tabla Delta estructurada y optimizada**.

--- 

Este es el final de este notebook. Continuamos en el siguiente.

In [0]:
DESCRIBE EXTENDED books

col_name,data_type,comment
book_id,string,null
title,string,null
author,string,null
category,string,null
price,double,null
,,
# Delta Statistics Columns,,
Column Names,"author, book_id, price, category, title",
Column Selection Method,first-32,
,,
